# Pods, Labels & the Object Model

## The Kubernetes object model

Open any Kubernetes manifest — Pod, Service, Deployment, ConfigMap, Secret, even a custom one — and you'll find the same five top-level keys. Memorise these now and you'll never feel lost in a new manifest again.

```
apiVersion: <which API group and version this object belongs to>
kind:       <what kind of object this is>
metadata:   <name, namespace, labels, annotations — who this object is>
spec:       <desired state — what you want the cluster to do>
status:     <actual state — what the cluster is currently observing>
```

A few things to lock in.

**`apiVersion` names the API group.** Core objects (Pod, Service, ConfigMap, Namespace, Node) use `v1` — the implicit "core" group. Newer objects use a group prefix: `apps/v1` for Deployments, `batch/v1` for Jobs and CronJobs, `networking.k8s.io/v1` for Ingress and NetworkPolicy, `rbac.authorization.k8s.io/v1` for RBAC. `kubectl api-resources` shows the full list.

**`kind` is the object type.** `Pod`, `Deployment`, `Service`, and so on. Always capitalised, singular.

**`metadata` is identifying information.** Every object has a `name` (unique within its namespace and kind), an optional `namespace` (defaults to `default`), `labels`, and `annotations`. Some fields are system-assigned: `uid`, `resourceVersion`, `creationTimestamp`.

**`spec` is what you want.** This is *your* contribution. You write `replicas: 3` because you want three replicas. You write `image: nginx:alpine` because you want that image.

**`status` is what is.** This is *the cluster's* contribution. You don't write it; the controllers fill it in. `phase: Running`, `podIP: 10.244.0.5`, `conditions: [...]` — all status fields. You read status to understand what actually happened.

### The reconcile loop

The relationship between `spec` and `status` is the whole game.

```
                  +-----------+
                  |   spec    |  desired state (you wrote it)
                  +-----+-----+
                        |
                        v
        +------------------------------------+
        |  controller                        |
        |  watches spec, observes the world, |
        |  takes action to close the gap     |
        +------------------+-----------------+
                           |
                           v
                  +-----------+
                  |  status   |  actual state (controller wrote it)
                  +-----------+
```

Every controller in Kubernetes runs the same loop. **Observe the world. Compare it to the desired spec. Take action to close the gap.** Forever. That's why we keep saying "declarative" — you describe the destination, and a thousand little control loops drive the cluster towards it.

When you `kubectl apply` a manifest, you're updating `spec`. Some controller, somewhere, notices and starts working. When the controller's done, `status` reflects reality.

## Anatomy of a Pod manifest

The smallest valid Pod manifest is four required keys:

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: hello
spec:
  containers:
    - name: app
      image: nginx:alpine
```

That manifest will create a pod and run it. But almost every pod you'll meet in the wild has more fields. Here's the realistic version, annotated:

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: web                      # required, unique within namespace + kind
  namespace: default             # optional, defaults to "default"
  labels:                        # key/value pairs you'll select on
    app: web
    tier: frontend
  annotations:                   # non-identifying metadata for tools
    owner: platform-team
spec:
  containers:
    - name: app                  # required, unique within the pod
      image: nginx:1.27-alpine   # required
      imagePullPolicy: IfNotPresent   # IfNotPresent | Always | Never
      ports:
        - containerPort: 80      # documentation; doesn't expose anything
          name: http             # optional, lets Services reference by name
          protocol: TCP          # TCP (default) | UDP | SCTP
      env:                       # environment variables for the container
        - name: GREETING
          value: hello
      command: ["nginx"]         # overrides image's ENTRYPOINT
      args: ["-g", "daemon off;"] # overrides image's CMD
      workingDir: /usr/share/nginx/html
  restartPolicy: Always          # Always (default) | OnFailure | Never
  serviceAccountName: default    # for in-cluster API access
  terminationGracePeriodSeconds: 30   # SIGTERM-to-SIGKILL window
```

A few things worth calling out before we run anything.

**`containerPort` is documentation, not a firewall.** Listing it doesn't open anything; the container is already free to listen on any port. The field is there so Services and Ingresses can reference it by name (`name: http`), and so humans skimming the manifest know what the container is supposed to expose.

**`command` overrides `ENTRYPOINT`, `args` overrides `CMD`.** Same semantics as Docker — same words too, just named differently. If you only set `args`, the image's `ENTRYPOINT` still runs but with your arguments.

**`imagePullPolicy` defaults aren't what you might expect.** If the image tag is `:latest` or omitted, the default is `Always` (pull every time). For any other tag, the default is `IfNotPresent` (only pull if missing). This is why "use specific tags" isn't just style — it changes behaviour.

**`restartPolicy` defaults to `Always` for bare pods.** We unpack the alternatives below.

**The pod is named, but a Deployment-managed pod gets an auto-generated suffix.** When we meet Deployments in notebook 03, pod names look like `web-7c9d8f8b8d-x2k5p` — the random tail is the controller's way of guaranteeing uniqueness across replicas.

In [ ]:
# Apply the pod manifest
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: Pod
metadata:
  name: web
  labels:
    app: web
    tier: frontend
spec:
  containers:
    - name: app
      image: nginx:1.27-alpine
      ports:
        - containerPort: 80
          name: http
      env:
        - name: GREETING
          value: hello
EOF
!echo '---'
# Read the object back — note how much status the API server filled in
!kubectl get pod web -o yaml | head -40

## Pod phases, conditions & container states

A pod's `status` has three different views of what's happening. They overlap, but each answers a different question.

### `status.phase` — the headline

One word, mutually exclusive:

| Phase | Meaning |
|---|---|
| `Pending` | Accepted by the cluster, but at least one container hasn't started — usually scheduling or image pull. |
| `Running` | Scheduled to a node, all containers created, at least one is running or starting or restarting. |
| `Succeeded` | All containers terminated successfully (`exit 0`) and will not be restarted. |
| `Failed` | All containers terminated, at least one failed. Will not be restarted under the default policy. |
| `Unknown` | The node hosting this pod has stopped reporting in. |

### `status.conditions` — the deeper view

A pod has several **conditions**, each either `True` or `False`. The ones you'll see:

- **`PodScheduled`** — the scheduler picked a node for the pod.
- **`Initialized`** — all init containers have completed successfully.
- **`ContainersReady`** — every container's readiness probe (if any) is passing.
- **`Ready`** — the pod itself is considered ready to receive traffic. Same as `ContainersReady` for single-container pods; different when readiness gates are used.

These conditions flip in order. A pod that's stuck — say, image pull is failing — will show `PodScheduled: True` but `Initialized: False` and `Ready: False`, and you can read the reason from the `message` field.

### `status.containerStatuses` — per-container state

For each container in the pod, you get a state with one of three forms:

- **`waiting`** — not running yet. `reason` tells you why: `ContainerCreating`, `ImagePullBackOff`, `CrashLoopBackOff`.
- **`running`** — up and running. `startedAt` tells you since when.
- **`terminated`** — finished. `exitCode`, `reason`, `startedAt`, and `finishedAt` are all populated.

And `restartCount` — how many times this container has been restarted by the kubelet. A growing `restartCount` is a red flag, even if the pod is "Running."

### The troubleshooting flow

When a pod is unhealthy, your first three commands are:

```
kubectl get pods                 # phase + restart count at a glance
kubectl describe pod <name>      # conditions, events, container reasons
kubectl logs <name> [--previous] # what the application said before it died
```

`describe` is the workhorse. Its `Events:` section at the bottom is the single richest source of "what just went wrong.

In [ ]:
# Headline view
!kubectl get pod web
!echo '---'
# Full describe with conditions and recent events
!kubectl describe pod web | sed -n '1,40p'
!echo '---'
# Pull out just the phase and the condition types
!kubectl get pod web -o jsonpath='Phase: {.status.phase}{"\n"}'
!kubectl get pod web -o jsonpath='Conditions: {range .status.conditions[*]}{.type}={.status} {end}{"\n"}'

## Restart policy

`spec.restartPolicy` controls what the kubelet does when a container in the pod exits. It applies to **all** containers in the pod — you can't set it per-container.

| Policy | Behaviour |
|---|---|
| `Always` *(default for Pods)* | Restart the container every time it exits, regardless of exit code. The right policy for long-running services. |
| `OnFailure` | Restart only if the container exited with a non-zero code. The right policy for batch work that should retry on failure but stop on success. |
| `Never` | Never restart. The right policy when you want one shot, success or failure. |

A few subtleties.

**Restart is the *kubelet* restarting the container, not Kubernetes rescheduling the pod.** Same node, same pod, same volumes — just a new container process. If the node itself dies, you need a higher-level controller (Deployment, ReplicaSet, DaemonSet) to bring the pod back somewhere else. A bare pod with `restartPolicy: Always` will *not* survive a node failure.

**CrashLoopBackOff.** When a container keeps crashing fast, the kubelet starts waiting longer between restarts — 10s, 20s, 40s, up to 5 minutes. That backoff state shows up in `kubectl get pods` as `CrashLoopBackOff`. It's not a separate phase — the pod is still `Running` from the cluster's perspective — but the container status is `Waiting` with that reason.

**For Jobs, the default is different.** When we meet Jobs in a later notebook, the pod template defaults to `restartPolicy: OnFailure`. The Job's own controller handles whole-pod failures separately.

In [ ]:
# restartPolicy: Never — one shot, success or failure
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: Pod
metadata:
  name: oneshot
spec:
  restartPolicy: Never
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c", "echo running; exit 1"]
EOF
!sleep 4
!kubectl get pod oneshot
!echo '---'
# restartPolicy: Always — the kubelet keeps restarting it
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: Pod
metadata:
  name: looper
spec:
  restartPolicy: Always
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c", "echo running; sleep 2; exit 1"]
EOF
!sleep 12
# Watch the RESTARTS column climb
!kubectl get pod looper

## Init containers

Sometimes a pod needs to do *something* before the main container starts — fetch a config file, run a database migration, wait for a dependency to be ready. That's what **init containers** are for.

Init containers run **one at a time, in order, to completion, before any main container starts.** If any init container fails, the pod's restart policy kicks in: `Always` or `OnFailure` will keep retrying; `Never` marks the whole pod failed.

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: web-with-init
spec:
  initContainers:
    - name: prepare-content
      image: busybox:1.36
      command: ["sh", "-c", "echo 'hello from init' > /shared/index.html"]
      volumeMounts:
        - name: html
          mountPath: /shared
  containers:
    - name: app
      image: nginx:alpine
      volumeMounts:
        - name: html
          mountPath: /usr/share/nginx/html
  volumes:
    - name: html
      emptyDir: {}
```

What this manifest does:

1. The pod is scheduled.
2. The init container `prepare-content` starts. It writes a file into the `html` volume.
3. The init container exits with code 0.
4. The main container `app` (nginx) starts, mounting the same volume — and serves the file the init container wrote.

Things to internalise about init containers:

- **They use the same container spec as main containers.** Same image fields, same env, same volume mounts. Anything you can do in a main container, you can do in an init container.
- **They're sequential.** Order matters; `initContainers` is a list, processed top-down.
- **They share the pod's volumes and network namespace.** That's how they pass work to the main containers — write to a shared `emptyDir`, write to a socket, whatever.
- **Their resources are *not* simply added to the main containers' resources.** The pod's effective request is computed using the max init container against the sum of main containers. Notebook 07 covers this properly.
- **The `Initialized` condition flips to `True` only after the last init container succeeds.**

In [ ]:
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: Pod
metadata:
  name: web-with-init
spec:
  initContainers:
    - name: prepare-content
      image: busybox:1.36
      command: ["sh", "-c", "echo 'hello from init' > /shared/index.html"]
      volumeMounts:
        - name: html
          mountPath: /shared
  containers:
    - name: app
      image: nginx:alpine
      volumeMounts:
        - name: html
          mountPath: /usr/share/nginx/html
  volumes:
    - name: html
      emptyDir: {}
EOF
!sleep 8
# Pod should be Running with Initialized: True
!kubectl get pod web-with-init
!echo '---'
# Read the file the init container left behind
!kubectl exec web-with-init -- cat /usr/share/nginx/html/index.html

## Multi-container pods — the sidecar pattern

Notebook 01 mentioned that the pod abstraction earns its keep when you have *helper* containers. Now we put that to use.

The **sidecar pattern**: a small companion container that adds a capability to the main container without changing the main container's image. Common examples:

- A log shipper that tails files written by the main container and forwards them to a central system.
- A service mesh proxy (Envoy in Istio, `linkerd-proxy`) that intercepts the main container's network traffic.
- A config reloader that watches a ConfigMap mount and signals the main container when it changes.

What makes the sidecar pattern work is the **shared volume** and the **shared network**.

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: web-with-sidecar
spec:
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c", "while true; do date >> /var/log/app.log; sleep 2; done"]
      volumeMounts:
        - name: logs
          mountPath: /var/log
    - name: log-shipper
      image: busybox:1.36
      command: ["sh", "-c", "tail -F /var/log/app.log"]
      volumeMounts:
        - name: logs
          mountPath: /var/log
  volumes:
    - name: logs
      emptyDir: {}
```

The `app` container writes timestamps to `/var/log/app.log`. The `log-shipper` container `tail -F`s the same file — different container, same file, because the volume is mounted in both. In a real system the shipper would forward those lines to Loki, Splunk, or CloudWatch.

You can read either container's stdout with `kubectl logs <pod> -c <container>`. The `-c` flag is required when the pod has more than one container — otherwise `kubectl logs` doesn't know which one you mean.

### Shared network — proof on `localhost`

If the main container listens on port 8080 and the sidecar wants to call it, the sidecar uses `localhost:8080`. No DNS, no IP lookups — they share the same network namespace. This is the same property that lets an Envoy sidecar transparently intercept traffic.

### When *not* to use a sidecar

Two containers don't have to share a pod just because they cooperate. Two **independent services that call each other over the network** belong in two **separate pods** behind a Service. Put them in the same pod only when they:

- must live on the same node (e.g. shared filesystem),
- have the same lifecycle (start together, die together),
- and share the pod's network or storage.

A common rookie mistake is bundling a web app and its database in one pod. They're independent services with independent scaling and lifecycle needs — they belong in separate pods, with the database fronted by its own Service.

In [ ]:
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: Pod
metadata:
  name: web-with-sidecar
spec:
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c", "while true; do date >> /var/log/app.log; sleep 2; done"]
      volumeMounts:
        - name: logs
          mountPath: /var/log
    - name: log-shipper
      image: busybox:1.36
      command: ["sh", "-c", "tail -F /var/log/app.log"]
      volumeMounts:
        - name: logs
          mountPath: /var/log
  volumes:
    - name: logs
      emptyDir: {}
EOF
!sleep 10
# Two containers in the pod
!kubectl get pod web-with-sidecar -o jsonpath='Containers: {.spec.containers[*].name}{"\n"}'
!echo '---'
# Read the shipper's stdout (the file it's tailing)
!kubectl logs web-with-sidecar -c log-shipper --tail=3

## Probes — liveness, readiness, startup

A container can be *running* but not *working*. It might be deadlocked, stuck in a slow garbage-collection pause, or waiting forever on a downstream service that will never come back. The container's process never exited, so the kubelet thinks all is well — but your users are seeing timeouts.

**Probes** let the kubelet ask the container "are you actually OK?" There are three kinds, each answering a different question.

### Liveness probe — "should I restart you?"

If a liveness probe fails repeatedly, the kubelet **kills and restarts the container** (subject to `restartPolicy`). Use it for "this container has wedged itself, only a restart will help" cases.

### Readiness probe — "should I send you traffic?"

If a readiness probe fails, the kubelet marks the container as **not ready**. The pod's `Ready` condition flips to `False`, and any Service in front of the pod stops sending it traffic. **The container is not restarted.** Use it for "I'm alive but I'm warming up, or I lost my DB connection, or I'm shedding load" cases.

### Startup probe — "are you finished booting yet?"

For containers that take a long time to start, a startup probe gives them a grace period. While it's still failing, **liveness and readiness probes are suspended**. Once it succeeds once, the startup probe is done and the other probes take over.

### Three ways to probe

Each probe type can use one of three mechanisms:

```yaml
livenessProbe:
  httpGet:                # HTTP GET; 2xx/3xx = pass, anything else = fail
    path: /healthz
    port: 8080
  initialDelaySeconds: 5
  periodSeconds: 10
  failureThreshold: 3
```

```yaml
readinessProbe:
  tcpSocket:              # just try to open a TCP connection
    port: 5432
```

```yaml
livenessProbe:
  exec:                   # run a command inside the container; exit 0 = pass
    command: ["sh", "-c", "test -f /tmp/healthy"]
```

The knobs you'll set most:

- `initialDelaySeconds` — wait this long before the first probe (give the container time to boot).
- `periodSeconds` — how often to probe.
- `failureThreshold` — how many consecutive failures before action is taken.
- `successThreshold` — how many consecutive passes before flipping back to OK (readiness only).
- `timeoutSeconds` — how long to wait per probe before counting it as a failure.

Defaults are reasonable for most apps. The CKA cares less about exact numbers and more about *which probe type to pick for which failure mode*.

In [ ]:
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: Pod
metadata:
  name: probed
spec:
  containers:
    - name: app
      image: nginx:alpine
      ports:
        - containerPort: 80
      readinessProbe:
        httpGet:
          path: /
          port: 80
        initialDelaySeconds: 2
        periodSeconds: 5
      livenessProbe:
        httpGet:
          path: /
          port: 80
        initialDelaySeconds: 10
        periodSeconds: 10
        failureThreshold: 3
EOF
!sleep 10
!kubectl get pod probed
!echo '---'
# Watch the conditions flip — ContainersReady and Ready should both be True
!kubectl get pod probed -o jsonpath='{range .status.conditions[*]}{.type}={.status} {end}{"\n"}'

## Labels — turning a flat list into a queryable set

Once you have more than a handful of pods, "list everything and look for it" stops scaling. **Labels** are how Kubernetes solves this. They're key/value pairs attached to objects in `metadata.labels`, designed for *identifying* and *selecting* groups of related objects.

```yaml
metadata:
  name: web-7c9d
  labels:
    app: web
    tier: frontend
    version: v2
    environment: prod
```

A few rules:

- **Keys are strings**, with an optional DNS-subdomain prefix: `app.kubernetes.io/name=web`. Prefixes are conventional namespacing — tools use them to avoid clashing.
- **Values are strings**, up to 63 characters, alphanumeric plus `-`, `_`, `.`.
- **An object can have any number of labels.** Add as many as make sense for your selection needs.
- **Labels are *identifying* metadata**, meant for selection. They are case-sensitive.

### Conventional labels (`app.kubernetes.io/*`)

The community settled on a recommended set of labels for any workload:

| Label | Meaning |
|---|---|
| `app.kubernetes.io/name` | The application name (`web`, `redis`, `payments`). |
| `app.kubernetes.io/instance` | A unique name for *this* deployment of the app (`payments-prod`). |
| `app.kubernetes.io/version` | The application version (`1.7.2`). |
| `app.kubernetes.io/component` | Component within the app (`api`, `worker`, `cache`). |
| `app.kubernetes.io/part-of` | Higher-level grouping (`checkout-platform`). |
| `app.kubernetes.io/managed-by` | What tool created this (`helm`, `kustomize`, `argocd`). |

You don't have to use these, but if you do, tools — dashboards, Helm, Argo CD — automatically understand your structure.

### Setting and changing labels

You can write labels in a manifest, or change them on the fly:

- `kubectl label pod web env=prod` — add a label.
- `kubectl label pod web env=staging --overwrite` — change a label.
- `kubectl label pod web env-` — remove a label (trailing `-`).
- `kubectl get pods --show-labels` — list pods with their full label sets.

Changing a label is more consequential than it looks. A controller selects pods by label; if you change a pod's labels, you might be *removing it from* its Deployment's set (so the Deployment will create a new one to replace it) or *adding it to* another's. A useful debugging trick: relabel a misbehaving pod out of its Deployment, leave it for inspection, and let the Deployment respawn its sibling.

In [ ]:
# Show the labels on the pods we've created so far
!kubectl get pods --show-labels
!echo '---'
# Add and change labels
!kubectl label pod web env=prod
!kubectl label pod web tier=web --overwrite
!kubectl get pod web --show-labels
!echo '---'
# Remove a label
!kubectl label pod web tier-
!kubectl get pod web --show-labels

## Label selectors

A **selector** is a small expression that filters objects by their labels. There are two flavours — **equality-based** (the simple one) and **set-based** (the powerful one).

### Equality-based

The everyday syntax. Three operators: `=`, `==` (same as `=`), and `!=`.

```
app=web                # label "app" equals "web"
tier!=frontend         # label "tier" is anything except "frontend"
app=web,env=prod       # AND — both must be true
```

Comma-separated terms mean AND. There is no OR — for OR you need set-based selectors.

```bash
kubectl get pods -l app=web
kubectl get pods -l 'app=web,env=prod'
kubectl get pods -l 'tier!=frontend'
```

### Set-based

The richer language. Four operators: `in`, `notin`, `exists` (just the key, no value), and `does-not-exist` (`!key`).

```
environment in (prod, staging)
tier notin (frontend, backend)
release                          # label "release" exists (any value)
!release                         # label "release" does not exist
```

```bash
kubectl get pods -l 'environment in (prod,staging)'
kubectl get pods -l '!canary'
```

You can combine multiple set-based terms with commas (AND). You can also mix equality- and set-based terms in one selector.

### Where selectors show up

Selectors are *everywhere* in Kubernetes:

- **`kubectl -l`** — interactive filtering.
- **Services** — `spec.selector` picks the pods the Service routes to.
- **Deployments, ReplicaSets, DaemonSets, Jobs** — `spec.selector` picks the pods they manage.
- **NetworkPolicies** — `podSelector` picks the pods the policy applies to.
- **PodDisruptionBudgets, HorizontalPodAutoscalers** — same idea.

Everywhere you'd want to refer to "a group of pods" without naming them individually, the answer is a label selector.

### Why this matters more than it looks

Notice what selectors are *not*: they're not foreign keys, not references by name, not edges in a graph. The connection between a Service and its pods is *loose* — the Service says "any pod with `app=web`" and Kubernetes resolves that *every* time, in real time. Add a pod with that label and it joins the set automatically. Remove the label and it falls out. This is the same loose coupling that makes rolling updates possible — the Service doesn't track individual pods, just "whichever ones match right now.

In [ ]:
# Add labels so we have something interesting to select
!kubectl label pod web-with-init app=web tier=frontend --overwrite
!kubectl label pod web-with-sidecar app=web tier=frontend --overwrite
!kubectl label pod probed app=web tier=frontend env=staging --overwrite
!echo '---'
# Equality
!kubectl get pods -l app=web
!echo '---'
# Set-based: env IN (prod, staging)
!kubectl get pods -l 'env in (prod,staging)'
!echo '---'
# Label exists (any value)
!kubectl get pods -l env

## Annotations — non-identifying metadata

Labels are for *selection*. **Annotations** are for *everything else you want to attach to an object*. Same key/value shape, lives at `metadata.annotations`, but with no constraint on length or character set, and no use by selectors.

```yaml
metadata:
  annotations:
    owner: platform-team@example.com
    description: "Front-door HTTP service for the public website"
    git-commit: abc123def
    kubectl.kubernetes.io/last-applied-configuration: |
      ...full prior manifest...
```

Use cases:

- **Human-readable notes** for whoever's reading the manifest next week.
- **Build metadata** — git commit, CI run URL, image digest at deploy time.
- **Tool configuration** — Ingress controllers, service meshes, and operators all read annotations to opt features on or off without changing the object's schema. E.g. `nginx.ingress.kubernetes.io/rewrite-target`.
- **Kubernetes' own bookkeeping** — `kubectl apply` stores its last-applied state in an annotation; that's how it computes a three-way merge on the next apply.

Rule of thumb: if you want to **find** this object later by some property, use a label. If you want to **describe** the object or carry data that some tool will read, use an annotation.

```bash
kubectl annotate pod web owner=platform-team
kubectl annotate pod web owner=ops --overwrite
kubectl annotate pod web owner-
```

## Namespaces — soft tenancy boundaries

A **namespace** is a way to scope object names. Two pods can both be called `web` if they're in different namespaces — `default/web` and `staging/web` are distinct objects. Most resource kinds are *namespaced*; a handful (Nodes, PersistentVolumes, StorageClasses, ClusterRoles) are *cluster-scoped*.

What a namespace gives you:

- **Name uniqueness scope** — names only need to be unique within the namespace+kind pair.
- **A target for RBAC** — you can grant a user access to "namespace `team-a`" without giving them the whole cluster.
- **A target for quotas and limits** — `ResourceQuota` and `LimitRange` apply per namespace.
- **A DNS suffix** — Services get a DNS name like `<service>.<namespace>.svc.cluster.local`.

What a namespace **does not** give you:

- **Network isolation.** By default, every pod can reach every other pod regardless of namespace. NetworkPolicy (notebook 09) is what isolates traffic.
- **Strong tenancy.** Namespaces are *soft* boundaries, suitable for separating teams or environments within one trusted org, not for hostile multi-tenancy.

### The built-in namespaces

A fresh cluster has four:

- **`default`** — where your stuff lands if you don't specify. Use it for learning; don't use it for real workloads.
- **`kube-system`** — Kubernetes' own components live here. CoreDNS, the kube-proxy DaemonSet, controller-manager, scheduler.
- **`kube-public`** — readable by everyone, even unauthenticated users. Rarely used.
- **`kube-node-lease`** — node heartbeats. Each node renews a Lease object here every few seconds.

### Switching namespaces

`-n <ns>` works on every command. To make it stick:

```bash
kubectl config set-context --current --namespace=team-a
```

That changes your *kubeconfig context* so subsequent commands default to `team-a`. A common shortcut is the `kubens` helper (part of `kubectx`) which gives you a menu to switch.

In [ ]:
# What namespaces exist on this cluster
!kubectl get namespaces
!echo '---'
# Kubernetes' own components live in kube-system
!kubectl get pods -n kube-system | head -10
!echo '---'
# Create a namespace and list pods across all of them
!kubectl create namespace demo 2>&1 || echo 'already exists'
!kubectl get pods -A | head -10

## Cleaning up

Delete everything this notebook created so the cluster's tidy for notebook 03:

```bash
kubectl delete pod web web-with-init web-with-sidecar probed oneshot looper
kubectl delete namespace demo
```

Or, more bluntly, every pod in the `default` namespace:

```bash
kubectl delete pods --all
```